# Autograd

Remember the last section in the lecture about **tensors**. By now you have hopefully drawn the conclusion that backpropagation, loss calculation, and gradient computation require continuous numbers (float),because the updates applied to weights and biases during training are extremely small. These sub‑integer adjustments make floating‑point precision necessary, which is why **float32** is the default data type for tensors in PyTorch.

This lecture describes how gradients are calculated in PyTorch. The method is straightforward and represents one of PyTorch’s major strengths: its built‑in gradient computation system **Autograd**, short for *automatic differentiation*. Autograd tracks operations on tensors and computes gradients automatically.

This system is activated through a single function argument:

- requires_grad

PyTorch treats tensors as *data* unless we explicitly tell it to treat them as *learnable parameters*. To do this, we must set:

requires_grad = True

**Note:** This setting is one of the most important in all of PyTorch.

When this setting is enabled, the message to the Autograd engine is:

- “This tensor is a parameter. Track every operation applied to it.”

PyTorch will then record all operations involving that tensor and build a computational graph. When .backward() is called, PyTorch uses this graph to compute gradients automatically.

In [11]:
import torch 

# First we create a tensor 
data_tensor = torch.tensor([[1., 2.], [3., 4.]])

parameter_tensor = torch.tensor([[1.0], [2.0]], requires_grad=True)

print(f"data_tensor requires_grad: {data_tensor.requires_grad}")
print(f"parameter_tensor requires_grad: {parameter_tensor.requires_grad}")

data_tensor requires_grad: False
parameter_tensor requires_grad: True


 ---
## Exercise 4 – Loss, Gradients and Backpropagation

By now you may have forgotten some details from the overview lecture. To understand what actually happens when PyTorch builds a computational graph and how it is used, complete the following exercise:

**Describe Loss, Gradients, Backpropagation and Optimizer how they are connected and what each of them does, in 3–4 sentences.**

 ---
## The Computational Graph

The **computational graph** is the structure PyTorch builds to keep track of how tensors are transformed during the forward pass. Every time you perform an operation on a tensor with requires_grad=True, PyTorch records that operation as a node in a graph. This graph represents the entire chain of computations that leads from your inputs, through intermediate operations, all the way to the final loss value.

 ---
## Why the computational graph exists
The purpose of the computational graph is to make **automatic differentiation** possible. To compute gradients, PyTorch must know:
- which tensors depend on which other tensors  
- which operations were applied  
- in what order those operations happened  

The graph encodes all of these dependencies so that PyTorch can apply the **chain rule** backwards during backpropagation.

 ---
## What the graph is made of
The graph consists of two types of nodes:
- **Tensor nodes**: store values (inputs, parameters, intermediate results, outputs, loss)
- **Function nodes**: store operations (e.g., addition, matrix multiplication, ReLU)

Each function node also stores the information needed to compute its gradient during backpropagation. This is why every tensor created from an operation has a grad_fn attached to it.

 ---
## How PyTorch uses the graph
When you call loss.backward(), PyTorch walks the graph **in reverse order**, starting from the loss. At each function node, it applies the chain rule to compute how much each parent tensor contributed to the loss. These contributions accumulate into the .grad attribute of every leaf tensor (typically your model parameters).

After backpropagation, PyTorch frees the graph by default to save memory.

 ---
## Summary
The computational graph is the internal blueprint of all operations that produced the loss. PyTorch builds it dynamically during the forward pass and uses it during backpropagation to compute gradients automatically. Without this graph, PyTorch would have no way to know how each parameter influenced the final loss.

All this amazing functionality is locked up with requires_grad = True.


 ---
## Understanding the Computational Graph (with example)

PyTorch builds a **computational graph** every time you perform operations on tensors with requires_grad=True.  
This graph records *how* each tensor was created, so that PyTorch can later compute gradients using backpropagation.

Below is a minimal example that shows exactly how the graph is built and used. For this we will use the 
- .grad_fn 

Every tensor in PyTorch has an attribute called grad_fn that tells you which operation created that tensor.

If the tensor was created by the user, it has no grad_fn
- Because it is a leaf tensor in the graph

If the tensor was created by an operation, i does have a grad_fn
- Because PyTorch must know how to compute its gradient

In [12]:
import torch

# Leaf tensors in the graph since it was created by user
w = torch.tensor([2.0], requires_grad=True) 
x = torch.tensor([5.0], requires_grad=True) 

# No grad_fn becouse it is not the result of an operation
print(w.grad_fn) # None
print(x.grad_fn) # None

# grad_fn becouse PyTorch needs to know how to compute 
# its gradient
y = w*x
print(y.grad_fn) # <MulBackward0 object> 

None
None


 ---
## Vizualising the computational graph 

Everytime you perform an operation:

- y = w*x
- loss = y.sum()

PyTorch creates a chain of grad_fn objects:

```Code
w (leaf) -----
               \
                * (MulBackward0) → y
               /
x (leaf) -----
```

Each arrow is a grad_fn link. This chain is the computational graph:
- y → sum (SumBackward0) → loss


 ---
## How grad_fn is used during backpropagation

When you call:
- loss.backward()

PyTorch:
- Starts at loss.grad_fn (SumBackward0 in our example)

- Calls its backward function

- Moves to the previous grad_fn (MulBackward0 in our example)

- Computes gradients using the chain rule

- Continues until it reaches all leaf tensors

- Stores the final gradients in .grad

So grad_fn is literally the mechanism that makes backpropagation possible.